# Sinh dataset SFT tiếng Việt cho ngành F&B

Notebook này sinh từng mẫu **tuần tự** (1 request/lần) và **append ngay** vào file output để tránh mất dữ liệu nếu bị ngắt.

**Thứ tự chạy:**
1. Cài deps
2. Kiểm tra kết nối Azure OpenAI
3. Load taxonomy + seed
4. Vòng lặp sinh mẫu (chạy lại bao nhiêu lần cũng được, sẽ tự skip mẫu đã có)

## 1. Cài đặt thư viện

In [48]:
# %pip install -q openai python-dotenv tenacity tqdm

## 2. Kiểm tra kết nối Azure OpenAI

Luôn reload `.env` mới nhất bằng `override=True`. Gửi 1 prompt "Xin chào" để xác nhận deployment hoạt động.

In [49]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI

# Tìm .env: ưu tiên thư mục notebook, fallback repo root
ENV_PATH = Path.cwd() / ".env"
if not ENV_PATH.exists():
    ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH, override=True)
print(f"Loaded .env: {ENV_PATH}")

AZURE_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
AZURE_KEY = os.environ["AZURE_OPENAI_API_KEY"]
OPENAI_API_VERSION = os.environ.get("OPENAI_API_VERSION", "2024-08-01-preview")
OPENAI_MODEL = os.environ.get("AZURE_DEPLOYMENT_NAME", "md-gpt-5.4-mini")

client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_KEY,
    api_version=OPENAI_API_VERSION,
)

Loaded .env: d:\Github\mcs-train-content-model\.env


In [50]:
_resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Xin chào, hãy trả lời ngắn gọn bằng tiếng Việt."}],
)
print(f"Endpoint   : {AZURE_ENDPOINT}")
print(f"Deployment : {OPENAI_MODEL}")
print(f"Api version: {OPENAI_API_VERSION}")
print(f"Reply      : {_resp.choices[0].message.content.strip()}")
print("\n✅ Kết nối Azure OpenAI thành công.")
 

Endpoint   : https://vqnhan-poc.openai.azure.com
Deployment : md-gpt-5.4-mini
Api version: 2024-12-01-preview
Reply      : Xin chào! Tôi sẽ trả lời ngắn gọn bằng tiếng Việt. Bạn cần mình hỗ trợ gì?

✅ Kết nối Azure OpenAI thành công.


## 3. Cấu hình taxonomy F&B + load seed

In [51]:
import json
import random

SCRIPT_DIR = Path.cwd()
SEED_FILE = SCRIPT_DIR / "marketing_social_media_dataset_vi.json"
OUTPUT_FILE = SCRIPT_DIR / "fnb_dataset_vi.json"

TARGET_COUNT = 500  # tổng số mẫu mong muốn trong OUTPUT_FILE
MERGE_SEED = False   # không dùng seed

SUB_SEGMENTS = [
    "Chuỗi cà phê tầm trung", "Cà phê đặc sản specialty", "Cà phê take-away",
    "Trà sữa", "Trà trái cây / matcha", "Nước ép & smoothie healthy",
    "Nhà hàng fine-dining", "Nhà hàng buffet", "Chuỗi lẩu nướng BBQ",
    "Quán nhậu / beer club", "Bia thủ công craft beer", "Rượu vang nhập khẩu",
    "Fast food gà rán / burger", "Pizza chuỗi", "Tiệm bánh ngọt bakery",
    "Kem / dessert shop", "Cloud kitchen giao hàng", "Chuỗi bún phở",
    "Cơm văn phòng / cơm tấm", "Đồ ăn vặt street food",
    "Snack đóng gói (bánh kẹo)", "Sản phẩm sữa / sữa chua",
    "Gia vị / nước chấm đóng chai", "Thực phẩm đông lạnh tiện lợi",
    "Đặc sản OCOP vùng miền", "Siêu thị mini / grocery",
    "Đồ uống đóng chai RTD", "Trà thảo mộc / nước detox",
    "Nhà hàng chay / thuần chay", "Cửa hàng tiện lợi 24/7",
]

WORKFLOW_STAGES = [
    "Chiến lược thương hiệu", "Định vị & naming sản phẩm mới",
    "Menu engineering & pricing", "Kế hoạch ra mắt sản phẩm mới",
    "Content calendar mạng xã hội", "Caption Facebook",
    "Caption Instagram", "Kịch bản video TikTok",
    "Kế hoạch livestream TikTok Shop", "Kế hoạch livestream Facebook",
    "Chiến lược KOL/KOC foodie", "Mời food reviewer & sự kiện trải nghiệm",
    "Quảng cáo Facebook Ads", "Quảng cáo TikTok Ads",
    "Quảng cáo Google Ads", "Quảng cáo GDN / banner",
    "Bài blog SEO", "Landing page khuyến mãi",
    "Email marketing", "SMS / Zalo OA broadcast",
    "Chương trình loyalty / membership", "Push notification app",
    "Khai trương cửa hàng mới", "Sự kiện activation tại điểm bán",
    "Chiến dịch Tết Nguyên Đán", "Chiến dịch Trung Thu",
    "Chiến dịch Valentine / 8-3 / 20-10", "Chiến dịch Giáng Sinh / Năm Mới",
    "Black Friday / Shopee 11-11 sale", "CSR & sustainability",
    "Crisis PR sự cố an toàn thực phẩm", "Phản hồi review tiêu cực",
    "Pitch dịch vụ B2B catering", "Tuyển dụng nhượng quyền franchise",
    "Employer branding tuyển bếp/barista", "A/B testing creative",
    "Báo cáo & phân tích hiệu quả", "Audience research & insight",
    "Packaging design brief", "OOH billboard ngã tư",
]

LENGTH_PROFILES = [
    ("ngắn gọn 50-100 từ", "Response súc tích, đi thẳng vào ý chính, không lan man."),
    ("trung bình 120-200 từ", "Response chi tiết vừa phải, có cấu trúc rõ ràng."),
    ("dài 250-400 từ", "Response chuyên sâu, đầy đủ KPI, phân bổ ngân sách, đo lường cụ thể."),
]

TONE_VARIANTS = [
    "chuyên nghiệp như agency",
    "thân thiện như freelancer",
    "kiểu founder startup",
    "kiểu director marketing dày dạn",
]

# Không dùng seed, AI tự generate
print("✅ Sẵn sàng sinh mẫu từ đầu.")

✅ Sẵn sàng sinh mẫu từ đầu.


## 4. Prompt template + hàm tiện ích

In [52]:
import hashlib
import re
from tenacity import retry, stop_after_attempt, wait_exponential

SYSTEM_PROMPT = """Bạn là chuyên gia marketing F&B Việt Nam với 10+ năm kinh nghiệm, đã làm cho Highlands, Phúc Long, Golden Gate, Masan. Bạn đang viết dữ liệu training SFT cho agent tạo nội dung Facebook marketing F&B.

YÊU CẦU TUYỆT ĐỐI:
1. Trả về DUY NHẤT 1 đối tượng JSON hợp lệ, không markdown, không giải thích, không ```json
2. Schema: {"instruction": "...", "input": "...", "response": "..."}
3. Tiếng Việt tự nhiên, dùng thuật ngữ marketing chuẩn (CPL, CTR, ROAS, GMV, AOV, retention...)
4. "instruction" là nhiệm vụ viết bài Facebook content, định vị rõ ràng thương hiệu/sản phẩm/offer/đối tượng và mục tiêu KPI.
5. "input" phải có ít nhất: Công ty, Đối tượng, Sản phẩm/dịch vụ, Giới hạn/Ngân sách, Offer/ưu đãi, Mục tiêu KPI cụ thể có số, Giai đoạn workflow, Kênh Facebook.
6. "response" phải là bài Facebook content hoàn chỉnh, copy/paste ready, có hook, nội dung chính, CTA, KPI cụ thể và con số VND/%, không chung chung.
7. Bối cảnh Việt Nam thật: tên thương hiệu/KOL/địa danh/nền tảng VN (Shopee, TikTok Shop, Zalo, ShopeeFood, GrabFood, Be, Momo, ZaloPay, Sơn Tùng, Trấn Thành, Khoai Lang Thang, Hà Linh, Tina Thảo Thi, Call Me Duy, Schannel...)
8. KHÔNG copy nguyên văn từ seed, phải biến tấu sáng tạo
9. Số tiền dùng VND (triệu/tỷ), không dùng USD"""

USER_TEMPLATE = """Tạo 1 mẫu training SFT marketing F&B Việt Nam cho content agent Facebook với cấu hình:

- Phân khúc F&B: {segment}
- Giai đoạn workflow: {workflow}
- Độ dài response: {length}
- Tông giọng: {tone}

{length_note}

Trả về DUY NHẤT 1 JSON object gồm:
- instruction: mô tả nhiệm vụ viết bài Facebook, định vị rõ ràng.
- input: bối cảnh thương hiệu, đối tượng, sản phẩm, ưu đãi, mục tiêu, ngân sách, kênh.
- response: bài Facebook hoàn chỉnh, copy/paste lên Facebook được ngay.

Không giải thích, không thêm text ngoài JSON."""


def sample_hash(sample: dict) -> str:
    text = sample.get("instruction", "") + sample.get("input", "")[:200]
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def extract_json(text: str) -> dict:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"Không tìm thấy JSON: {text[:200]}")
    return json.loads(text[start : end + 1])


def validate(sample: dict) -> dict:
    if not isinstance(sample, dict):
        raise ValueError("Không phải dict")
    for key in ("instruction", "input", "response"):
        if key not in sample or not isinstance(sample[key], str):
            raise ValueError(f"Thiếu/sai field: {key}")
        if len(sample[key].strip()) < 20:
            raise ValueError(f"Field {key} quá ngắn")
    if "công ty" not in sample["input"].lower():
        raise ValueError("Input thiếu phần Công ty")
    return {k: sample[k].strip() for k in ("instruction", "input", "response")}


@retry(stop=stop_after_attempt(1), wait=wait_exponential(multiplier=2, min=2, max=15))
def generate_one(segment: str, workflow: str) -> dict:
    length_label, length_note = random.choice(LENGTH_PROFILES)
    tone = random.choice(TONE_VARIANTS)
    user_msg = USER_TEMPLATE.format(
        segment=segment,
        workflow=workflow,
        length=length_label,
        length_note=length_note,
        tone=tone,
    )
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.9,
        top_p=0.95,
    )
    return validate(extract_json(resp.choices[0].message.content))


print("✅ Đã định nghĩa hàm sinh mẫu.")

✅ Đã định nghĩa hàm sinh mẫu.


In [53]:
# Test generate_one với 1 mẫu
try:
    test_sample = generate_one("Trà sữa", "Caption Facebook")
    print("✅ Test thành công!")
    print("Instruction:", test_sample["instruction"][:100])
    
    print("Input:", test_sample["input"][:100])
    print("Response:", test_sample["response"][:100])
except Exception as e:
    print(f"❌ Lỗi test: {e}")
    import traceback
    traceback.print_exc()

✅ Test thành công!
Instruction: Viết caption Facebook cho thương hiệu trà sữa theo giọng founder startup: ngắn gọn, thẳng ý, có chất
Input: Công ty: Mộc Tea Lab (trà sữa nội địa Việt Nam)
Đối tượng: Gen Z 18-24 và dân văn phòng trẻ 25-30 tạ
Response: Founder nói thật: trà sữa ngon không cần phải đắt. Mộc Tea Lab vừa ra mắt Trà sữa nướng trân châu đe


## 5. I/O an toàn — load output hiện có & append từng mẫu

Mỗi mẫu sinh ra sẽ được ghi vào file ngay lập tức (atomic write qua file tạm). Có thể dừng và chạy lại bất cứ lúc nào — script sẽ tiếp tục từ chỗ đang dở.

In [43]:
import tempfile


def load_output() -> list:
    if OUTPUT_FILE.exists():
        with OUTPUT_FILE.open("r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_output_atomic(data: list) -> None:
    """Ghi file qua tmp + rename để tránh corrupt khi bị ngắt giữa chừng."""
    fd, tmp_path = tempfile.mkstemp(
        dir=str(OUTPUT_FILE.parent), prefix=".tmp_", suffix=".json"
    )
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        os.replace(tmp_path, OUTPUT_FILE)
    except Exception:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise


current = load_output()
print(f"Output hiện có: {len(current)} mẫu trong {OUTPUT_FILE.name}")

Output hiện có: 500 mẫu trong fnb_dataset_vi.json


## 6. Vòng lặp sinh mẫu (tuần tự, append từng mẫu)

Chạy lại cell này bao nhiêu lần cũng được. Mỗi mẫu xong sẽ flush vào file `fnb_dataset_vi.json` ngay.

In [44]:
from tqdm.auto import tqdm

samples = load_output()
seen_hashes = {sample_hash(s) for s in samples}

# Nếu MERGE_SEED và output đang rỗng → cộng số seed vào target hiệu dụng
effective_target = TARGET_COUNT
if MERGE_SEED:
    # các seed đã được append sẵn ở lần đầu tiên (xem cell tiếp theo nếu muốn)
    pass

# Sinh ma trận task đa dạng
task_matrix = [(s, w) for s in SUB_SEGMENTS for w in WORKFLOW_STAGES]
random.shuffle(task_matrix)

remaining = effective_target - len(samples)
print(f"Hiện có: {len(samples)} | Mục tiêu: {effective_target} | Cần sinh thêm: {remaining}")

if remaining <= 0:
    print("✅ Đã đủ mẫu, không sinh thêm.")
else:
    pbar = tqdm(total=remaining, desc="Sinh mẫu")
    task_idx = 0
    error_count = 0
    while len(samples) < effective_target:
        seg, wf = task_matrix[task_idx % len(task_matrix)]
        task_idx += 1
        try:
            new_sample = generate_one(seg, wf)
            h = sample_hash(new_sample)
            if h in seen_hashes:
                pbar.set_postfix_str(f"trùng, skip ({seg[:15]})")
                continue
            seen_hashes.add(h)
            samples.append(new_sample)
            save_output_atomic(samples)  # flush ngay sau mỗi mẫu
            pbar.update(1)
            pbar.set_postfix_str(f"{seg[:18]} / {wf[:18]}")
            error_count = 0
        except Exception as e:
            error_count += 1
            pbar.write(f"⚠️ Lỗi ({seg[:20]} / {wf[:20]}): {str(e)[:120]}")
            if error_count >= 10:
                pbar.write("❌ Quá nhiều lỗi liên tiếp, dừng. Kiểm tra rate limit / quota.")
                break
    pbar.close()

print(f"\n✅ Tổng số mẫu trong {OUTPUT_FILE.name}: {len(samples)}")

Hiện có: 500 | Mục tiêu: 500 | Cần sinh thêm: 0
✅ Đã đủ mẫu, không sinh thêm.

✅ Tổng số mẫu trong fnb_dataset_vi.json: 500


## 7. (Tuỳ chọn) Gộp seed vào đầu file output

Nếu muốn 20 seed tiếng Việt nằm chung trong file training final, chạy cell này 1 lần.

In [45]:
if MERGE_SEED:
    # Không dùng seed
    print("Bỏ qua (MERGE_SEED=False)")
else:
    print("Bỏ qua (MERGE_SEED=False)")

Bỏ qua (MERGE_SEED=False)


## 8. (Tuỳ chọn) Xem thử 3 mẫu vừa sinh

In [46]:
data = load_output()
print(f"Tổng: {len(data)} mẫu\n")
for i, s in enumerate(data[-3:], 1):
    print(f"=== Mẫu cuối #{i} ===")
    print("Instruction:", s["instruction"])
    print("Input      :", s["input"][:200], "..." if len(s["input"]) > 200 else "")
    print("Response   :", s["response"][:300], "..." if len(s["response"]) > 300 else "")
    print()

Tổng: 500 mẫu

=== Mẫu cuối #1 ===
Instruction: Viết bài Facebook marketing cho chuỗi bún phở theo brief Packaging design brief, định vị thương hiệu là lựa chọn bữa sáng/bữa trưa tiện lợi, chuẩn vị, dễ nhận diện tại điểm bán và trên Facebook. Mục tiêu là tăng CTR vào bài giới thiệu bao bì mới và kéo khách ghé cửa hàng/đặt món. Tông giọng chuyên nghiệp như agency, ngắn gọn, rõ CTA, có KPI cụ thể.
Input      : Công ty: Chuỗi bún phở Phở Nhà Mình. Đối tượng: dân văn phòng 22-35 tuổi tại TP.HCM và Hà Nội, ưu tiên ăn nhanh, sạch, tiện. Sản phẩm/dịch vụ: bao bì mới cho combo bún phở mang đi, đồng bộ nhận diện t ...
Response   : Phở Nhà Mình ra mắt bao bì mới: sạch hơn, nhận diện rõ hơn, giữ nhiệt tốt hơn cho từng tô bún phở mang đi. Dành cho dân văn phòng cần bữa trưa nhanh mà vẫn chuẩn vị. Từ 15/5, combo phở bò tái nạm giảm ngay 20% khi check-in hoặc đặt qua ShopeeFood/GrabFood. Mục tiêu chiến dịch 7 ngày: CTR 4,5%, 1.200 ...

=== Mẫu cuối #2 ===
Instruction: Viết bài Facebook marketing cho